# 🏗️ PHASE 3: SCOPE & TIMELINE VISUALS (SUPREME THEME)
## Focus: Detailed Scope Tables & Floating Gantt Charts

In [ ]:
!pip install python-pptx pydantic -q

In [ ]:
import json
import base64
import os
from typing import List, Optional, Literal, Union, Dict, Any
from pydantic import BaseModel
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from IPython.display import display, HTML

# ==========================================
# 🎨 1. THE SUPREME DESIGN SYSTEM (UNCHANGED)
# ==========================================
class Colors:
    BG = [250, 250, 250]            # Clean White/Grey
    TITLE = [0, 0, 0]               # Black Titles
    TEXT = [60, 60, 60]             # Dark Grey Text
    ACCENT_PRIMARY = [0, 51, 102]   # Navy (Main Headers)
    ACCENT_SECONDARY = [225, 226, 246] # Lavender (Light Fills)
    TIMELINE_AXIS = [100, 100, 100] # Grey Axis
    # Timeline Bar Colors
    BAR_BLUE = [0, 112, 192]
    BAR_ORANGE = [237, 125, 49]
    BAR_PURPLE = [112, 48, 160]

class Fonts:
    SERIF = "Times New Roman"
    SANS = "Arial"

def get_rgb(c): return RGBColor(c[0], c[1], c[2])

def add_header(slide, title, subtitle=None):
    # Standard Header
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(12), Inches(1))
    p = tb.text_frame.paragraphs[0]
    p.text = title
    p.font.name = Fonts.SERIF
    p.font.bold = True
    p.font.size = Pt(36)
    p.font.color.rgb = get_rgb(Colors.TITLE)
    
    if subtitle:
        sb = slide.shapes.add_textbox(Inches(0.5), Inches(1.0), Inches(12), Inches(0.5))
        p = sb.text_frame.paragraphs[0]
        p.text = subtitle
        p.font.name = Fonts.SANS
        p.font.size = Pt(12)
        p.font.color.rgb = get_rgb(Colors.TITLE)

# ==========================================
# 📝 2. DATA MODELS
# ==========================================

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: str

class ScopeItem(BaseModel):
    system: str
    description: List[str]

class SlideScope(BaseModel):
    type: Literal["scope"]
    title: str
    subtitle: str
    items: List[ScopeItem]

class TimelineTask(BaseModel):
    label: str
    start_week: float
    duration: float
    row: int          # Controls vertical height (1 = top, 2 = below, etc)
    color: str        # "blue", "orange", "purple"

class SlideTimeline(BaseModel):
    type: Literal["timeline"]
    title: str
    total_weeks: int
    tasks: List[TimelineTask]

class PresentationConfig(BaseModel):
    filename: str
    slides: List[Union[SlideCover, SlideScope, SlideTimeline]]

# ==========================================
# 🖌️ 3. RENDERERS
# ==========================================

def render_cover(prs, data: SlideCover):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_header(slide, data.title)
    
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(3), Inches(12), Inches(2))
    p = tb.text_frame.paragraphs[0]
    p.text = data.subtitle
    p.font.size = Pt(24)

def render_scope(prs, data: SlideScope):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_header(slide, data.title, data.subtitle)

    # Table Configuration
    rows = len(data.items) + 1 # +1 for Header
    cols = 2
    top = Inches(1.8)
    left = Inches(0.5)
    width = Inches(12.33)
    height = Inches(5.0)
    
    shape = slide.shapes.add_table(rows, cols, left, top, width, height)
    table = shape.table
    
    # Set Column Widths (Left is narrow, Right is wide)
    table.columns[0].width = Inches(3.0)
    table.columns[1].width = Inches(9.33)

    # --- HEADER ROW ---
    h1 = table.cell(0, 0)
    h1.text = "System/Processes"
    h1.text_frame.paragraphs[0].font.bold = True
    h1.text_frame.paragraphs[0].font.color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
    
    h2 = table.cell(0, 1)
    h2.text = "Description"
    h2.text_frame.paragraphs[0].font.bold = True
    h2.text_frame.paragraphs[0].font.color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
    
    # --- DATA ROWS ---
    for i, item in enumerate(data.items):
        row_idx = i + 1
        
        # Left Cell (Navy Block)
        c1 = table.cell(row_idx, 0)
        c1.fill.solid()
        c1.fill.fore_color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
        c1.text = item.system
        p = c1.text_frame.paragraphs[0]
        p.font.color.rgb = get_rgb([255, 255, 255])
        p.font.bold = True
        p.font.size = Pt(14)
        c1.vertical_anchor = MSO_ANCHOR.MIDDLE
        
        # Right Cell (Light Content)
        c2 = table.cell(row_idx, 1)
        c2.fill.solid()
        c2.fill.fore_color.rgb = get_rgb(Colors.ACCENT_SECONDARY)
        
        tf = c2.text_frame
        tf.clear() # Clear default paragraph
        for point in item.description:
            p = tf.add_paragraph()
            p.text = f"• {point}"
            p.font.size = Pt(11)
            p.font.color.rgb = get_rgb(Colors.TITLE)
            p.space_after = Pt(6)

def render_timeline(prs, data: SlideTimeline):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    add_header(slide, data.title)

    # Axis Settings
    margin_left = 1.0
    graph_width = 11.0
    axis_y = 6.0
    unit_w = graph_width / data.total_weeks
    
    # 1. Draw Axis Bar
    axis = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(margin_left), Inches(axis_y), Inches(graph_width), Inches(0.4))
    axis.fill.solid()
    axis.fill.fore_color.rgb = get_rgb(Colors.TIMELINE_AXIS)
    axis.text_frame.text = " " # Empty text

    # 2. Draw Weeks Labels
    for w in range(data.total_weeks + 1):
        x = margin_left + (w * unit_w)
        tb = slide.shapes.add_textbox(Inches(x), Inches(axis_y + 0.1), Inches(unit_w), Inches(0.3))
        p = tb.text_frame.paragraphs[0]
        p.text = f"Week {w}"
        p.font.size = Pt(10)
        p.font.color.rgb = get_rgb([255, 255, 255])
        
    # 3. Draw Floating Tasks
    base_h = 0.5
    spacing = 0.1
    
    for task in data.tasks:
        # Calculate Position
        bar_x = margin_left + (task.start_week * unit_w)
        bar_w = task.duration * unit_w
        # Row 1 is highest up, Row 4 is lowest (closest to axis)
        # Let's say Row 1 starts at Y=2.0
        bar_y = 1.5 + (task.row * (base_h + spacing))
        
        # Draw Bar
        bar = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(bar_x), Inches(bar_y), Inches(bar_w), Inches(base_h))
        bar.fill.solid()
        
        # Color Mapping
        if task.color == "blue": bar.fill.fore_color.rgb = get_rgb(Colors.BAR_BLUE)
        elif task.color == "orange": bar.fill.fore_color.rgb = get_rgb(Colors.BAR_ORANGE)
        elif task.color == "purple": bar.fill.fore_color.rgb = get_rgb(Colors.BAR_PURPLE)
        else: bar.fill.fore_color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
        
        bar.line.fill.background()
        
        # Label
        p = bar.text_frame.paragraphs[0]
        p.text = task.label
        p.font.size = Pt(10)
        p.font.bold = True

# ==========================================
# 🚀 4. USER DATA (JSON INPUT)
# ==========================================

input_json = {
  "filename": "Supreme_Phase3.pptx",
  "slides": [
    {
      "type": "cover",
      "title": "Project Scope & Timeline",
      "subtitle": "Phase 3 Deliverables"
    },
    # --- NEW: SCOPE SLIDE (Matches Image 1) ---
    {
      "type": "scope",
      "title": "Scope (1/2)",
      "subtitle": "The scope of the project is to develop an application for self serving KIOSK Terminals.",
      "items": [
        {
          "system": "Consumer Kiosk Platform",
          "description": [
            "Touchscreen-based UI for intuitive user experience.",
            "Integration with gaming backend for real-time data.",
            "Guest checkout functionality (Name & Mobile only)."
          ]
        },
        {
          "system": "Payment Integration",
          "description": [
            "Cash Transactions: Paper and polymer note acceptance.",
            "Card Transactions: Visa/Mastercard SDK integration.",
            "Seamless Wallet: Top up gaming accounts via Cash/Card."
          ]
        },
        {
          "system": "Ticket Verification",
          "description": [
            "Support for users to scan tickets to verify winnings.",
            "Print receipts and game tickets."
          ]
        },
        {
          "system": "Real-Time Monitoring",
          "description": [
            "Live analytics dashboard for transaction summaries.",
            "Automated alerts & error tracking for admins."
          ]
        }
      ]
    },
    # --- NEW: TIMELINE SLIDE (Matches Image 2) ---
    {
      "type": "timeline",
      "title": "Timeline for Kiosk Application Development",
      "total_weeks": 6,
      "tasks": [
        # Row 1 (Top)
        { "label": "Requirements and UI Design", "start_week": 0, "duration": 1.5, "row": 1, "color": "blue" },
        # Row 2
        { "label": "Development of Custom Kiosk App (Lottery/Games)", "start_week": 1.0, "duration": 4.0, "row": 2, "color": "orange" },
        # Row 3
        { "label": "Integration with SDKs (Print/Pay)", "start_week": 2.5, "duration": 3.0, "row": 3, "color": "blue" },
        # Row 4 (Parallel)
        { "label": "Updates to CMS", "start_week": 3.0, "duration": 2.0, "row": 4, "color": "purple" },
        { "label": "Reports Dev", "start_week": 5.2, "duration": 0.8, "row": 4, "color": "purple" },
        # Milestone
        { "label": "UAT & GO LIVE", "start_week": 5.8, "duration": 1.0, "row": 5, "color": "purple" }
      ]
    }
  ]
}

# ==========================================
# 🏁 5. EXECUTE
# ==========================================

try:
    print("⚙️ Generating Phase 3 Deck...")
    config = PresentationConfig(**input_json)
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    for slide_data in config.slides:
        if slide_data.type == "cover":
            render_cover(prs, slide_data)
        elif slide_data.type == "scope":
            render_scope(prs, slide_data)
        elif slide_data.type == "timeline":
            render_timeline(prs, slide_data)

    prs.save(config.filename)
    
    with open(config.filename, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{config.filename}" style="background: #0078D4; color: white; padding: 10px 20px; border-radius: 5px; text-decoration: none; font-weight: bold;">⬇️ DOWNLOAD PHASE 3 PPT</a>'
    display(HTML(link))
    print("✅ Done.")

except Exception as e:
    print(f"❌ Error: {e}")